In [ ]:
# 1. Configurar GPU
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 2. Instalar dependencias
!pip install -q tf2onnx onnx

In [ ]:
# 3. Descargar dataset PlantVillage desde GitHub
import os
import urllib.request
import zipfile

DATASET_URL = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"
DATASET_ZIP = "plantvillage.zip"

if not os.path.exists("PlantVillage-Dataset-master"):
    print("Descargando dataset PlantVillage (~250MB)...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)
    print("Extrayendo...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(".")
    os.remove(DATASET_ZIP)
    print("Dataset listo!")
else:
    print("Dataset ya existe")

!ls PlantVillage-Dataset-master/raw/

In [ ]:
# 4. Preparar generadores de datos
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_DIR = "PlantVillage-Dataset-master/raw/color"
IMG_SIZE = 224
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

NUM_CLASSES = len(train_gen.class_indices)
print(f"\n{NUM_CLASSES} clases, {train_gen.samples} imagenes de entrenamiento")

In [ ]:
# 5. Crear modelo MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print(f"Modelo creado: {model.count_params():,} parametros")

In [ ]:
# 6. Entrenar Fase 1: Solo capas nuevas
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=2)
]

print("Fase 1: Entrenando capas de clasificacion...")
history1 = model.fit(
    train_gen,
    epochs=5,
    validation_data=val_gen,
    callbacks=callbacks
)
print(f"\nAccuracy Fase 1: {history1.history['val_accuracy'][-1]:.2%}")

In [ ]:
# 7. Fine-tuning: Descongelar ultimas capas
print("Fase 2: Fine-tuning...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    epochs=5,
    validation_data=val_gen,
    callbacks=callbacks
)

final_acc = history2.history['val_accuracy'][-1]
print(f"\nAccuracy Final: {final_acc:.2%}")

In [ ]:
# 8. Guardar modelo en formato SavedModel
model.save('plant_disease_savedmodel', save_format='tf')
print("Modelo guardado en SavedModel")

In [ ]:
# 9. Convertir a ONNX
!python -m tf2onnx.convert --saved-model plant_disease_savedmodel --output plant_disease_model.onnx --opset 13

import os
size = os.path.getsize('plant_disease_model.onnx') / (1024*1024)
print(f"\nModelo ONNX: {size:.1f} MB")

In [ ]:
# 10. Crear archivo de labels para Android
import json

class_names = list(train_gen.class_indices.keys())
labels = []

translations = {
    'healthy': 'Saludable',
    'Early_blight': 'Tizon temprano',
    'Late_blight': 'Tizon tardio',
    'Bacterial_spot': 'Mancha bacteriana',
    'Leaf_Mold': 'Moho foliar',
    'Septoria_leaf_spot': 'Septoriosis',
    'Spider_mites Two-spotted_spider_mite': 'Acaros',
    'Target_Spot': 'Mancha diana',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Virus rizado amarillo',
    'Tomato_mosaic_virus': 'Virus del mosaico',
    'powdery_mildew': 'Oidio',
    'Black_rot': 'Podredumbre negra',
    'Cedar_apple_rust': 'Roya del manzano',
    'Esca_(Black_Measles)': 'Esca',
    'Leaf_blight_(Isariopsis_Leaf_Spot)': 'Tizon foliar',
    'Common_rust_': 'Roya comun',
    'Northern_Leaf_Blight': 'Tizon norteno',
    'Cercospora_leaf_spot Gray_leaf_spot': 'Mancha gris',
    'Haunglongbing_(Citrus_greening)': 'Huanglongbing',
    'Apple_scab': 'Sarna del manzano'
}

for i, name in enumerate(class_names):
    parts = name.split('___')
    crop = parts[0].replace('_', ' ')
    disease_raw = parts[1] if len(parts) > 1 else 'healthy'
    display = translations.get(disease_raw, disease_raw.replace('_', ' '))
    
    labels.append({
        "id": i,
        "name": name,
        "crop": crop,
        "display": display
    })

with open('plant_disease_labels.json', 'w', encoding='utf-8') as f:
    json.dump(labels, f, indent=2, ensure_ascii=False)

print(f"Labels guardados: {len(labels)} clases")
for l in labels[:5]:
    print(f"  {l['id']}: {l['crop']} - {l['display']}")

In [ ]:
# 11. Descargar archivos
from google.colab import files

print("Descargando archivos...")
files.download('plant_disease_model.onnx')
files.download('plant_disease_labels.json')

print("\n=== INSTRUCCIONES ===")
print("En tu PC, ejecuta:")
print("")
print("cd ~/HuaweiProject/AgroChat_Project/tools")
print("./mindspore-lite-2.3.1-linux-x64/tools/converter/converter_lite \\")
print("    --fmk=ONNX \\")
print("    --modelFile=plant_disease_model.onnx \\")
print("    --outputFile=plant_disease_model")
print("")
print("cp plant_disease_model.ms ../app/src/main/assets/")
print("cp plant_disease_labels.json ../app/src/main/assets/")